# Category-Level Privacy Patterns

Analysis for category-level z-score comparisons of privacy metrics.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

metrics_results = Path("../data/mhealth_apps_metrics.csv")
if not metrics_results.exists():
    raise FileNotFoundError(
        "Could not find `mhealth_apps_metrics.csv` in the notebook folder or at ../data/."
    )

REGION_MAP = {
    "us": "North America", "ca": "North America", "gl":"North America",
    "mx": "Latin America", "br": "Latin America", "ar": "Latin America", 
    "cl": "Latin America",
    "de": "Europe", "fr": "Europe", "gb": "Europe", 
    "pl": "Europe",
    "tr": "Middle East", "il": "Middle East", "sa": "Middle East", 
    "ae": "Middle East",
    "in": "Asia", "jp": "Asia", "sg": "Asia", 
    "kr": "Asia",
    "za": "Africa", "mz": "Africa", "ng": "Africa", 
    "ke": "Africa",
    "au": "Oceania", "nz": "Oceania", 
    "pg": "Oceania"
}

COUNTRY_LABEL_MAP = {
    "us": "United States", "ca": "Canada", "gl":"Greenland",
    "mx": "Mexico", "br": "Brazil", "ar": "Argentina",
    "cl": "Chile",
    "de": "Germany", "fr": "France", "gb": "Great Britain",
    "pl": "Poland",
    "tr": "Turkey", "il": "Israel", "sa": "Saudi Arabia",
    "ae": "United Arab Emirates",
    "in": "India", "jp": "Japan", "sg": "Singapore",
    "kr": "South Korea",
    "za": "South Africa", "mz": "Mozambique", "ng": "Nigeria",
    "ke": "Kenya",
    "au": "Australia", "nz": "New Zealand",
    "pg": "Papua New Guinea"
}

REGION_ORDER = [
    "North America", "Latin America", "Europe",
    "Middle East", "Asia", "Africa", "Oceania"
]

region_palette = plt.cm.tab10(np.linspace(0, 1, len(REGION_ORDER)))
REGION_COLORS = {region: region_palette[i] for i, region in enumerate(REGION_ORDER)}
category_palette = plt.cm.tab20(np.linspace(0, 1, 20))


In [ ]:
import pandas as pd

df = pd.read_csv(metrics_results, low_memory=False)
print("Raw shape:", df.shape)

required_metrics = ["ADII", "DGI", "PCLR", "AS"]
missing_required = [c for c in required_metrics if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required metric columns: {missing_required}")

analysis_df = df.copy()

print("Analysis shape:", analysis_df.shape)
print("Unique apps:", analysis_df["app_id"].nunique())

display_cols = [
    "app_id", "country", "country_label", "region", "category",
    "ADII", "DGI", "PCLR", "AS",
    "observed_count", "disclosed_count", "missing_count", "misleading_count"
]
display_cols = [c for c in display_cols if c in analysis_df.columns]

In [ ]:

summary = pd.DataFrame({
    "non_null": analysis_df[["ADII", "DGI", "PCLR", "AS"]].notna().sum(),
    "mean": analysis_df[["ADII", "DGI", "PCLR", "AS"]].mean(),
    "median": analysis_df[["ADII", "DGI", "PCLR", "AS"]].median(),
    "std": analysis_df[["ADII", "DGI", "PCLR", "AS"]].std(),
    "min": analysis_df[["ADII", "DGI", "PCLR", "AS"]].min(),
    "max": analysis_df[["ADII", "DGI", "PCLR", "AS"]].max(),
}).round(4)
summary


In [ ]:

country_level = analysis_df.copy()

metric_agg = {
    "ADII": "mean",
    "DGI": "mean",
    "PCLR": "mean",
    "AS": "mean",
}

meta_agg = {
    "country_label": "nunique",
    "region": lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
    "category": lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
    "num_permissions": "mean",
    "num_dangerous_permissions": "mean",
    "num_trackers": "mean",
    "downloads_int": "mean",
    "average_score": "mean",
}

agg_dict = {}
for k, v in {**metric_agg, **meta_agg}.items():
    if k in analysis_df.columns:
        agg_dict[k] = v

app_level = analysis_df.groupby("app_id", as_index=False).agg(agg_dict)
app_level = app_level.rename(columns={"country_label": "country_count"})
app_level_all = app_level.copy()

if "PCLR" not in app_level_all.columns:
    if "app_country_PCLR" in analysis_df.columns:
        pclr_fallback = (
            analysis_df.groupby("app_id", as_index=False)["app_country_PCLR"]
            .mean()
            .rename(columns={"app_country_PCLR": "PCLR"})
        )
        app_level_all = app_level_all.merge(pclr_fallback, on="app_id", how="left")
    elif {"app_country_pre_sensitive_instances", "app_country_total_sensitive_instances"}.issubset(analysis_df.columns):
        pclr_fallback = (
            analysis_df.groupby("app_id", as_index=False)[
                ["app_country_pre_sensitive_instances", "app_country_total_sensitive_instances"]
            ]
            .mean()
        )
        denom = pclr_fallback["app_country_total_sensitive_instances"].replace(0, np.nan)
        pclr_fallback["PCLR"] = pclr_fallback["app_country_pre_sensitive_instances"] / denom
        app_level_all = app_level_all.merge(pclr_fallback[["app_id", "PCLR"]], on="app_id", how="left")

print("Country-level rows:", len(country_level))
print("App-level rows:", len(app_level_all))
print("App-level metric columns:", [c for c in ["ADII", "DGI", "PCLR", "AS"] if c in app_level_all.columns])
app_level = app_level_all.copy()




## Figure — Category heatmap of normalized metric means

A heatmap is useful for comparing relative patterns across categories and metrics at the same time.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

# Ensure directory exists
output_dir = Path("../figures")
output_dir.mkdir(parents=True, exist_ok=True)

required_cols = ["category", "ADII", "DGI", "PCLR", "AS"]
missing = [c for c in required_cols if c not in app_level_all.columns]

if missing:
    raise KeyError(f"app_level_all is missing required columns: {missing}")

# --- category-level aggregation (one row per app) ---
category_metric = (
    app_level_all.groupby("category", dropna=False)[["ADII", "DGI", "PCLR", "AS"]]
    .mean()
)

category_metric = category_metric.loc[category_metric.index.notna()]

category_counts = (
    app_level_all.groupby("category", dropna=False)["app_id"]
    .nunique()
    .loc[category_metric.index]
)

# z-score normalize by metric for comparability
heatmap_data = (category_metric - category_metric.mean()) / category_metric.std(ddof=0)
heatmap_data = heatmap_data.replace([np.inf, -np.inf], np.nan).fillna(0)

# Sort categories by the z-scored average privacy burden, not the raw
# average. ADII's raw scale (roughly 600-1300) is ~1000x larger than
# DGI/PCLR/AS's [0,1] range, so a raw mean.axis(1) is effectively just the
# ADII ranking -- sorting on the normalized data reflects all four metrics.
sort_order = heatmap_data.mean(axis=1).sort_values(ascending=False).index
category_metric = category_metric.loc[sort_order]
heatmap_data = heatmap_data.loc[sort_order]
category_counts = category_counts.loc[sort_order]

# Clip values for better visual balance
heatmap_data = heatmap_data.clip(-2, 2)

fig, ax = plt.subplots(figsize=(4, 2.5))

# Disable grid explicitly
ax.grid(False)

im = ax.imshow(
    heatmap_data.values,
    aspect="auto",
    cmap="RdYlBu_r",
    vmin=-2,
    vmax=2,
    alpha=0.95
)

ax.set_xticks(np.arange(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns, rotation=0, fontsize=8)
ax.set_yticks(np.arange(len(heatmap_data.index)))
ax.set_yticklabels(
    [f"{cat} (n={category_counts.loc[cat]})" for cat in heatmap_data.index],
    fontsize=8,
)

# ax.set_title(
#     "Category-level Relative Privacy Posture (Normalized)",
#     fontsize=13,
#     weight="bold"
# )

# Annotate with raw mean values
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        val = heatmap_data.iloc[i, j]
        raw = category_metric.iloc[i, j]
        ax.text(
            j, i,
            f"{raw:.2f}",
            ha="center",
            va="center",
            fontsize=8,
            color="black" if abs(val) < 1 else "white"
        )

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Normalized score (z-score)")

plt.tight_layout()
plt.savefig(output_dir / "category_privacy_heatmap.png", dpi=600, bbox_inches="tight")
plt.show()